# **Clasificación de Razas de Perros con Deep Learning**

## Objetivo del proyecto

Este proyecto explora distintas estrategias de clasificación de imágenes utilizando redes neuronales convolucionales (CNNs) y técnicas de *transfer learning* sobre un dataset de razas de perros.

El objetivo principal es comparar diferentes enfoques de modelado y evaluar cómo afectan al rendimiento final del sistema de clasificación.

A lo largo del proyecto se desarrollan y comparan los siguientes enfoques:

- Modelo base completamente conectado
- CNN diseñada desde cero
- Optimización mediante regularización y ajuste de hiperparámetros
- Transfer Learning con InceptionV3
- Técnicas de Data Augmentation
- AutoML con AutoKeras

## Tecnologías utilizadas

- Python
- TensorFlow / Keras
- AutoKeras
- NumPy
- Pandas
- Matplotlib
- Seaborn
- Scikit-learn

## Dataset

El dataset utilizado en este proyecto se encuentra publicado en Hugging Face.

El conjunto de datos contiene más de 12.000 imágenes distribuidas en 74 razas distintas de perros y se utiliza para entrenar modelos de clasificación multiclase mediante Deep Learning.

## Fuente del dataset

https://huggingface.co/datasets/lpastor75/dog-breed-classification

## Descarga del dataset

El dataset se encuentra publicado en Hugging Face y se descarga automáticamente utilizando `huggingface_hub`.

## Instalación de dependencias

Este notebook instala automáticamente las librerías necesarias para garantizar reproducibilidad en cualquier entorno (Colab, local o Kaggle).

In [ ]:
# Instalación de dependencias
!pip install -r requirements.txt

In [ ]:
import os
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import tensorflow as tf
import autokeras as ak

from sklearn.model_selection import train_test_split
from sklearn import metrics

plt.style.use("ggplot")

# ============================================
# REPRODUCIBILIDAD
# ============================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================
# CONFIGURACIÓN GLOBAL
# ============================================

IMG_SIZE_BASE = (100, 120, 3)
IMG_SIZE_TRANSFER = (299, 299, 3)

BATCH_SIZE = 128
EPOCHS = 10

In [ ]:
# Instalación de dependencias necesarias

!pip install -q huggingface_hub

In [ ]:
# Descarga del dataset desde Hugging Face

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="lpastor75/dog-breed-classification",
    repo_type="dataset",
    local_dir="dog-breed-dataset"
)

# Estructura del dataset

Tras la descarga, la estructura local del dataset será:
```
dog-breed-dataset/
│
├── README.md
├── train.csv
├── valid.csv
├── test.csv
└── dog-images.zip
```
Las imágenes se encuentran comprimidas en un único archivo ZIP para optimizar almacenamiento y transferencia.

In [ ]:
# Descompresión del dataset

import zipfile

zip_path = "dog-breed-dataset/dog-images.zip"

extract_path = "dog-breed-dataset/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset descomprimido correctamente.")

## Carga de imágenes y etiquetas

Se generan automáticamente:
- las rutas de las imágenes
- las clases disponibles
- las etiquetas numéricas necesarias para entrenamiento

In [ ]:
# Directorio raíz de imágenes
dataset_path = "dog-breed-dataset/dog-images"

# Clases disponibles
classes = sorted([
    folder_name
    for folder_name in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, folder_name))
])

# Diccionarios de conversión
num_to_label = {
    i: c
    for i, c in enumerate(classes)
}

label_to_num = {
    c: i
    for i, c in enumerate(classes)
}

# Rutas completas de imágenes
image_paths = np.array(
    glob.glob(
        os.path.join(dataset_path, "**", "*.jpg"),
        recursive=True
    )
)

# Etiquetas numéricas
image_labels = np.array([
    label_to_num[
        os.path.basename(os.path.dirname(path))
    ]
    for path in image_paths
])

print(f"Número total de imágenes: {len(image_paths)}")
print(f"Número total de clases: {len(classes)}")

## Reproducibilidad

El dataset se descarga directamente desde Hugging Face para garantizar:
- reproducibilidad
- versionado
- trazabilidad de datos
- portabilidad del proyecto

## Exploración visual del dataset

Antes de entrenar los modelos, se realiza una inspección visual rápida del dataset para verificar:

- diversidad de imágenes
- variabilidad de tamaños
- diferencias entre clases
- calidad general de los datos

Este paso es importante para detectar posibles problemas de calidad o sesgos antes del entrenamiento.

In [ ]:
def show_images(img_paths, n_images=25):

    fig = plt.figure(figsize=(15, 15))

    random_indices = np.random.randint(
        len(img_paths),
        size=n_images
    )

    for i in range(n_images):

        fig.add_subplot(5, 5, i + 1)

        plt.axis("off")

        img = Image.open(img_paths[random_indices[i]])

        plt.imshow(img)

    plt.tight_layout()
    plt.show()


show_images(image_paths)

# **Distribución de clases**

Se analiza el número de imágenes disponibles por raza para comprobar si el dataset presenta desbalanceo entre clases.

In [ ]:
os.makedirs("images", exist_ok=True)

targets = [
    len(
        glob.glob(
            os.path.join(dataset_path, c, "*.jpg")
        )
    )
    for c in classes
]

plt.figure(figsize=(20, 6))

plt.bar(classes, targets)

plt.xticks(rotation=90)

plt.title("Distribución de imágenes por clase")

plt.xlabel("Clase")

plt.ylabel("Número de imágenes")

plt.savefig("images/class_distribution.png", bbox_inches="tight")

plt.show()

## División del dataset

El dataset se divide en tres subconjuntos:

- entrenamiento
- validación
- test

La partición se realiza manteniendo la distribución original de clases mediante muestreo estratificado.

In [ ]:
# Mezcla aleatoria
shuffler = np.random.permutation(len(image_paths))

image_paths = image_paths[shuffler]
image_labels = image_labels[shuffler]

# División train/test
x_train_valid, x_test, y_train_valid, y_test = train_test_split(
    image_paths,
    image_labels,
    test_size=0.10,
    random_state=SEED,
    stratify=image_labels
)

# División train/validation
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_valid,
    y_train_valid,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_valid
)

print(f"Train samples: {x_train.shape[0]}")
print(f"Validation samples: {x_valid.shape[0]}")
print(f"Test samples: {x_test.shape[0]}")

## Pipeline de carga de datos

Se implementa un pipeline basado en `tf.data.Dataset` para optimizar:

- lectura de imágenes
- preprocesamiento
- batching
- rendimiento durante entrenamiento

Este enfoque es más eficiente y escalable para proyectos de Deep Learning.

In [ ]:
def read_image(image_path, label):

    contents = tf.io.read_file(image_path)

    img = tf.image.decode_jpeg(
        contents,
        channels=3
    )

    img = tf.cast(img, tf.float32)

    img /= 255.0

    return img, label


def resize_image(img, label, target_size):

    resized_img = tf.image.resize(
        img,
        target_size
    )

    return resized_img, label


def get_dataset(
    image_paths,
    image_labels,
    target_size,
    batch_size,
    prep_func=None
):

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, image_labels)
    )

    dataset = dataset.map(
        read_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.map(
        lambda x, y: resize_image(x, y, target_size),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if prep_func is not None:

        dataset = dataset.map(
            lambda x, y: (x * 255.0, y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

        dataset = dataset.map(
            lambda x, y: (prep_func(x), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# **Modelo baseline**

Se construye un modelo inicial sencillo para establecer una referencia base de rendimiento.

Este modelo permitirá comparar posteriormente arquitecturas convolucionales más avanzadas y técnicas de transferencia de aprendizaje.

In [ ]:
train_dataset = get_dataset(
    x_train,
    y_train,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

valid_dataset = get_dataset(
    x_valid,
    y_valid,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

test_dataset = get_dataset(
    x_test,
    y_test,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=IMG_SIZE_BASE),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        1,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        len(classes),
        activation="softmax"
    )
])

model.summary()

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs=2,
    validation_data=valid_dataset
)

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_dataset
)

print(f"Test accuracy: {test_accuracy:.4f}")

## Evolución del entrenamiento

Se analiza la evolución de:

- función de pérdida
- accuracy en entrenamiento
- accuracy en validación

para detectar posibles problemas de:
- underfitting
- overfitting
- convergencia

In [ ]:
plt.figure(figsize=(14, 5))

# Loss
plt.subplot(1, 2, 1)

plt.plot(history.history["loss"], label="Train Loss")

plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Loss")

plt.legend()

# Accuracy
plt.subplot(1, 2, 2)

plt.plot(history.history["accuracy"], label="Train Accuracy")

plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Accuracy")

plt.legend()

plt.show()

## Matriz de confusión

La matriz de confusión permite identificar:

- clases con mayor tasa de error
- confusiones frecuentes entre razas
- comportamiento global del modelo

In [ ]:
predictions = model.predict(test_dataset)

y_pred = np.argmax(predictions, axis=1)

conf_matrix = metrics.confusion_matrix(
    y_true=y_test,
    y_pred=y_pred
)

df_cm = pd.DataFrame(
    conf_matrix,
    index=classes,
    columns=classes
)

plt.figure(figsize=(20, 20))

sns.heatmap(
    df_cm,
    cmap="Blues"
)

plt.title("Confusion Matrix")

plt.show()

---

# **Entrenamiento de una CNN desde cero**

En esta sección se desarrolla una Red Neuronal Convolucional (CNN) para clasificación multiclase de razas de perro.

El objetivo es construir un modelo base entrenado completamente desde cero y analizar su comportamiento antes de aplicar técnicas más avanzadas como regularización, transfer learning o data augmentation.

### Arquitectura del modelo

La red sigue una arquitectura inspirada en modelos tipo VGG:

- Tamaño de entrada: `(100, 120, 3)`
- Capa convolucional con 32 filtros y activación ReLU
- Capa MaxPooling
- Capa convolucional con 64 filtros y activación ReLU
- Capa MaxPooling
- Capa fully connected con 1024 neuronas
- Capa de salida Softmax

### Configuración del entrenamiento

| Parámetro | Valor |
|---|---|
| Optimizador | Adam |
| Learning Rate | 0.001 |
| Función de pérdida | Sparse Categorical Crossentropy |
| Batch Size | 128 |
| Epochs | 10 |

---

In [ ]:
# Configuración de datasets

train_dataset = get_dataset(
    x_train,
    y_train,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

valid_dataset = get_dataset(
    x_valid,
    y_valid,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

test_dataset = get_dataset(
    x_test,
    y_test,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

In [ ]:
# Definición del modelo CNN

cnn_model = tf.keras.Sequential([

    tf.keras.Input(shape=IMG_SIZE_BASE),

    tf.keras.layers.Conv2D(
        32,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(
        pool_size=(2, 2)
    ),

    tf.keras.layers.Conv2D(
        64,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(
        pool_size=(2, 2)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        1024,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        len(classes),
        activation="softmax"
    )
])

cnn_model.summary()

In [ ]:
# Compilación del modelo

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Entrenamiento

history_cnn = cnn_model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS
)

In [ ]:
# Evaluación final sobre test

test_loss, test_accuracy = cnn_model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Análisis del entrenamiento

Las siguientes gráficas muestran la evolución de las métricas de entrenamiento y validación durante el proceso de optimización.

Se monitorizan dos indicadores principales:

- Accuracy
- Loss

Estas curvas permiten analizar el comportamiento del modelo y detectar posibles problemas de sobreajuste.

---

In [ ]:
# Visualización del entrenamiento

plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)

plt.plot(
    history_cnn.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_cnn.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)

plt.plot(
    history_cnn.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_cnn.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Matriz de confusión

La matriz de confusión permite analizar en detalle el rendimiento del modelo para cada una de las clases del problema.

Este análisis facilita identificar:

- Razas correctamente clasificadas
- Clases con mayor nivel de confusión
- Posibles problemas de desbalanceo

---

In [ ]:
# Predicciones sobre el conjunto de test

predictions = cnn_model.predict(test_dataset)

y_pred = np.argmax(predictions, axis=1)
y_true = y_test

# Matriz de confusión

conf_matrix = metrics.confusion_matrix(
    y_true=y_true,
    y_pred=y_pred
)

# Visualización

df_cm = pd.DataFrame(
    conf_matrix,
    index=classes,
    columns=classes
)

plt.figure(figsize=(20, 20))

sns.heatmap(
    df_cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.show()

## Resultados y conclusiones

La CNN base consigue aprender patrones visuales relevantes y obtiene un rendimiento razonable sobre el dataset.

Sin embargo, las curvas de entrenamiento y validación muestran síntomas claros de sobreajuste:

- El accuracy de entrenamiento continúa aumentando
- El accuracy de validación se estabiliza rápidamente
- La loss de validación deja de mejorar tras varias epochs

Este comportamiento sugiere que la complejidad del modelo es elevada en relación con el tamaño del dataset disponible.

En las siguientes secciones se introducirán técnicas adicionales para mejorar la capacidad de generalización del modelo:

- Regularización mediante Dropout
- Data Augmentation
- Transfer Learning con modelos preentrenados

---

---

## **Optimización de hiperparámetros y regularización**

En este experimento se incorporan técnicas de regularización para reducir el sobreajuste detectado en la CNN inicial.

La nueva arquitectura introduce capas Dropout y una mayor profundidad en la extracción de características.

### Mejoras introducidas

- Bloques convolucionales adicionales
- Regularización mediante Dropout
- Mayor capacidad de extracción de features

### Arquitectura actualizada

- Tamaño de entrada: `(100, 120, 3)`
- Conv2D → MaxPooling
- Dropout (0.5)
- Conv2D → MaxPooling
- Conv2D → MaxPooling
- Dropout (0.5)
- Dense de 1024 neuronas
- Dropout (0.2)
- Capa Softmax de salida

---

In [ ]:
# Modelo CNN regularizado con Dropout

regularized_model = tf.keras.Sequential([

    tf.keras.Input(shape=IMG_SIZE_BASE),

    tf.keras.layers.Conv2D(
        32,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(
        pool_size=(2, 2),
        padding="same"
    ),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Conv2D(
        64,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(
        pool_size=(2, 2),
        padding="same"
    ),

    tf.keras.layers.Conv2D(
        64,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(
        pool_size=(2, 2),
        padding="same"
    ),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        1024,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(
        len(classes),
        activation="softmax"
    )
])

regularized_model.summary()

In [ ]:
# Compilación

regularized_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Entrenamiento

history_regularized = regularized_model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS
)

In [ ]:
test_dataset = get_dataset(
    x_test,
    y_test,
    IMG_SIZE_BASE[:-1],
    BATCH_SIZE
)

In [ ]:
# Evaluación final sobre test

test_loss, test_accuracy = regularized_model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Visualización del entrenamiento

plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)

plt.plot(
    history_regularized.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_regularized.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)

plt.plot(
    history_regularized.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_regularized.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predicciones sobre el conjunto de test

predictions = regularized_model.predict(test_dataset)

y_pred = np.argmax(predictions, axis=1)
y_true = y_test

# Matriz de confusión

conf_matrix = metrics.confusion_matrix(
    y_true=y_true,
    y_pred=y_pred
)

# Visualización

df_cm = pd.DataFrame(
    conf_matrix,
    index=classes,
    columns=classes
)

plt.figure(figsize=(20, 20))

sns.heatmap(
    df_cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.show()

## Resultados y conclusiones

La incorporación de capas Dropout mejora ligeramente la estabilidad del entrenamiento y reduce parcialmente el sobreajuste respecto al modelo inicial.

Aun así, el rendimiento en validación continúa estabilizándose de forma temprana, lo que indica que el tamaño del dataset sigue siendo una limitación importante.

Este experimento demuestra la importancia de equilibrar:

- Complejidad del modelo
- Tamaño del dataset
- Nivel de regularización

En la siguiente sección se utilizará Transfer Learning con InceptionV3 para aprovechar características previamente aprendidas sobre ImageNet.

---

# **Transfer Learning con InceptionV3**

## Objetivo

En esta fase del proyecto se implementa una estrategia de **Transfer Learning** utilizando el modelo preentrenado **InceptionV3**.  
El objetivo es aprovechar el conocimiento previamente adquirido sobre el dataset **ImageNet** para mejorar el rendimiento del clasificador de razas de perros.

A diferencia de las arquitecturas entrenadas desde cero, los modelos preentrenados permiten:

- Reducir significativamente el tiempo de entrenamiento
- Obtener mejores métricas con datasets medianos
- Minimizar problemas de sobreajuste
- Aprovechar características visuales ya aprendidas

Para este experimento:

- Se congela la base convolucional de InceptionV3
- Se añade una capa `Dropout` para regularización
- Se incorpora una nueva capa de clasificación adaptada al número de razas del dataset

---

## Configuración del modelo

| Parámetro | Valor |
|---|---|
| Arquitectura base | InceptionV3 |
| Input shape | `(299, 299, 3)` |
| Pesos preentrenados | `ImageNet` |
| Pooling | `avg` |
| Dropout | `0.5` |
| Optimizador | Adam |
| Learning Rate | `0.001` |
| Epochs | `10` |
| Batch Size | `128` |

In [ ]:
# Configuración de datasets para InceptionV3

train_dataset = get_dataset(
    x_train,
    y_train,
    IMG_SIZE_TRANSFER[:-1],
    BATCH_SIZE,
    tf.keras.applications.inception_v3.preprocess_input
)

valid_dataset = get_dataset(
    x_valid,
    y_valid,
    IMG_SIZE_TRANSFER[:-1],
    BATCH_SIZE,
    tf.keras.applications.inception_v3.preprocess_input
)

test_dataset = get_dataset(
    x_test,
    y_test,
    IMG_SIZE_TRANSFER[:-1],
    BATCH_SIZE,
    tf.keras.applications.inception_v3.preprocess_input
)

In [ ]:
# Carga del modelo preentrenado

base_model = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

# Congelación de pesos
for layer in base_model.layers:
    layer.trainable = False

# Construcción del clasificador final

x = tf.keras.layers.Dropout(0.5)(base_model.output)

outputs = tf.keras.layers.Dense(
    len(classes),
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs=base_model.input,
    outputs=outputs
)

# Compilación

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Entrenamiento del modelo

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS
)

In [ ]:
# Evaluación sobre test set

test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Análisis del entrenamiento

A continuación se visualiza la evolución de las métricas principales durante el proceso de entrenamiento:

- Accuracy
- Validation Accuracy
- Loss
- Validation Loss

Estas gráficas permiten detectar posibles problemas de:
- Sobreajuste
- Infraentrenamiento
- Estabilidad del modelo

In [ ]:
# Evolución del entrenamiento

plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()

# Loss
plt.subplot(1, 2, 2)

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.tight_layout()
plt.savefig("images/training_curves_inceptionv3.png", bbox_inches="tight")
plt.show()

## Matriz de confusión

La matriz de confusión permite analizar el comportamiento del modelo clase por clase, identificando:

- Razas correctamente clasificadas
- Clases con mayor confusión
- Posibles sesgos del dataset
- Categorías visualmente similares

In [ ]:
# Predicciones

predictions = model.predict(test_dataset)

y_pred = np.argmax(predictions, axis=1)
y_true = y_test

# Confusion Matrix

conf_matrix = metrics.confusion_matrix(
    y_true=y_true,
    y_pred=y_pred
)

df_cm = pd.DataFrame(
    conf_matrix,
    index=classes,
    columns=classes
)

# Visualización

plt.figure(figsize=(20, 20))

sns.heatmap(
    df_cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.savefig("images/confusion_matrix_inceptionv3.png", bbox_inches="tight")
plt.show()

## Resultados

El enfoque basado en **Transfer Learning** obtiene el mejor rendimiento de todos los experimentos realizados en el proyecto.

### Observaciones principales

- La convergencia es rápida y estable
- El modelo generaliza correctamente sobre validación y test
- El sobreajuste se reduce considerablemente
- Se alcanza una precisión significativamente superior frente a modelos CNN entrenados desde cero

La utilización de pesos preentrenados sobre ImageNet resulta especialmente efectiva para este problema debido a la similitud entre ambos dominios visuales.

Este experimento demuestra cómo las arquitecturas preentrenadas permiten construir soluciones robustas incluso cuando el tamaño del dataset es limitado.

# **Data Augmentation**

## Objetivo

En este experimento se aplican técnicas de **Data Augmentation** para incrementar artificialmente la variabilidad del dataset durante el entrenamiento.

El objetivo principal es:

- Mejorar la capacidad de generalización
- Reducir el sobreajuste
- Hacer el modelo más robusto frente a variaciones visuales

Las transformaciones aplicadas incluyen:

- Horizontal Flip
- Vertical Flip
- Random Contrast

Estas operaciones se ejecutan dinámicamente durante el entrenamiento mediante capas de preprocesado de TensorFlow.

In [ ]:
# Configuración de datasets

train_dataset = get_dataset(x_train, y_train, IMG_SIZE_BASE[:-1], BATCH_SIZE)

valid_dataset = get_dataset(x_valid, y_valid, IMG_SIZE_BASE[:-1], BATCH_SIZE)

test_dataset = get_dataset(x_test, y_test, IMG_SIZE_BASE[:-1], BATCH_SIZE)

In [ ]:
# CNN con Data Augmentation

model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=IMG_SIZE_BASE),

    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomContrast(0.2),

    tf.keras.layers.Conv2D(
        32,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

    tf.keras.layers.Conv2D(
        64,
        (5, 5),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        1024,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        len(classes),
        activation="softmax"
    )
])

# Compilación

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Entrenamiento

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS
)

In [ ]:
# Evaluación

test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Resultados

La estrategia de Data Augmentation consigue una ligera mejora en generalización respecto al modelo CNN base.

Sin embargo, el impacto sigue siendo limitado frente al rendimiento obtenido mediante Transfer Learning.

### Observaciones

- El sobreajuste sigue presente
- El entrenamiento es más estable
- Algunas transformaciones no aportan mejoras relevantes
- El `vertical flip` puede introducir ejemplos poco realistas para este caso de uso

Posibles líneas futuras de mejora:

- Rotaciones suaves
- Ajustes de brillo
- Gaussian Noise
- Random Zoom
- Combinación con Dropout

# **Conclusiones**

## Comparativa de experimentos

A lo largo del proyecto se han evaluado múltiples enfoques para clasificación de razas de perros:

| Modelo | Resultado |
|---|---|
| CNN básica | Bajo rendimiento y sobreajuste |
| CNN + Dropout | Mejora parcial |
| CNN + Data Augmentation | Generalización moderadamente mejor |
| InceptionV3 Transfer Learning | Mejor rendimiento global |

---

## Modelo recomendado para producción

El modelo basado en **InceptionV3 + Transfer Learning** sería la opción elegida para un entorno productivo.

### Motivos principales

- Mayor accuracy en validación y test
- Mejor capacidad de generalización
- Entrenamiento más estable
- Menor sobreajuste
- Excelente relación rendimiento/coste computacional

Además, Transfer Learning demuestra ser una estrategia altamente eficiente cuando se dispone de datasets de tamaño medio y problemas similares a ImageNet.

---

# Próximas mejoras

Posibles líneas futuras para evolucionar el proyecto:

- Fine-Tuning parcial desbloqueando capas superiores
- Aumento del dataset
- Experimentación con EfficientNet o ConvNeXt
- Exportación del modelo a TensorFlow Lite
- Despliegue mediante API REST o Streamlit
- Monitorización de inferencia en producción

---

## Limitaciones

- Dataset relativamente pequeño para algunas razas
- Algunas clases presentan similitud visual elevada
- El modelo no ha sido optimizado para inferencia en tiempo real
- No se ha realizado fine-tuning parcial de capas superiores

Nota:
Debido a incompatibilidades entre versiones recientes de Keras y AutoKeras en Google Colab, la sección de Autokeras ha sido eliminada ya que puede requerir versiones específicas de TensorFlow/Keras.